# JALARI — Visi AI: Latih Model Klasifikasi Limbah Organik MBG
Transfer learning **MobileNetV2** → 4 kelas: **buah, sayur, daging, tulang**. Output siap dipakai langsung oleh `public/js/vision.js` (TensorFlow.js, drop-in).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mostoples/jalari/blob/main/notebooks/jalari_vision_train.ipynb)

## Cara pakai
1. **Runtime → Change runtime type → GPU** (T4 gratis).
2. Jalankan **sel 1** (install). ⚠️ **PENTING:** `tensorflowjs` menurunkan versi TensorFlow, jadi setelah sel 1 selesai **lakukan Runtime → Restart session SEKALI**. Ini mencegah error `undefined symbol` / TF rusak di tengah sesi.
3. Setelah restart: **Runtime → Run all**. Seluruh notebook kini berjalan di versi TF yang konsisten (training + konversi).
4. Saat diminta: upload **`kaggle.json`** (wajib — buah/sayur/daging) dan *(opsional)* **`tulang.zip`** 50–150 foto tulang (kalau dilewati → model 3 kelas).

Di akhir, **`jalari_model.zip`** otomatis terunduh. Ekstrak isinya ke **`public/model/`** di repo JALARI — selesai, tanpa ubah kode.

In [ ]:
# 1) Dependencies  —  JALANKAN SEL INI, LALU: Runtime -> Restart session (SEKALI), baru Run all.
!pip install -q tensorflowjs split-folders kaggle
import importlib
for m in ['tensorflowjs','splitfolders']:
    try: importlib.import_module(m); print(m, 'OK')
    except Exception as e: print('(akan beres setelah restart)', m, '->', type(e).__name__)
print('\n>>> SEKARANG: Runtime -> Restart session, lalu Runtime -> Run all. <<<')

In [ ]:
# 2) Upload kaggle.json (WAJIB)
import os
from google.colab import files
print('Pilih file kaggle.json ...')
files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json && chmod 600 /root/.kaggle/kaggle.json
print('kaggle.json siap.')

In [ ]:
# 3) Unduh dataset (buah+sayur, daging)
!kaggle datasets download -d kritikseth/fruit-and-vegetable-image-recognition -p /content/dl/fv --unzip -q
!kaggle datasets download -d crowww/meat-quality-assessment-based-on-deep-learning -p /content/dl/meat --unzip -q
print('Dataset terunduh.')

In [ ]:
# 4) (OPSIONAL) Kelas 'tulang' — upload tulang.zip (boleh batal/skip)
import os, zipfile, glob
os.makedirs('/content/tulang', exist_ok=True)
try:
    from google.colab import files
    up = files.upload()
    for fn in up:
        if fn.lower().endswith('.zip'):
            zipfile.ZipFile(fn).extractall('/content/tulang')
except Exception as e:
    print('Lewati tulang:', e)
print('Jumlah foto tulang:', len([p for p in glob.glob('/content/tulang/**/*', recursive=True) if p.lower().endswith(('.jpg','.jpeg','.png'))]))

In [ ]:
# 5) Susun dataset 4 kelas terpadu + seimbangkan
import os, glob, shutil, random
random.seed(42)
FRUITS = {'apple','banana','grapes','kiwi','lemon','mango','orange','pear','pineapple','pomegranate','watermelon'}
EXT = ('.jpg','.jpeg','.png','.bmp','.webp')
def imgs(d): return [p for p in glob.glob(d+'/**/*', recursive=True) if p.lower().endswith(EXT)]
buckets = {'buah':[], 'sayur':[], 'daging':[], 'tulang':[]}
for p in imgs('/content/dl/fv'):
    cls = os.path.basename(os.path.dirname(p)).lower().strip()
    buckets['buah' if cls in FRUITS else 'sayur'].append(p)
buckets['daging'] = imgs('/content/dl/meat')
buckets['tulang'] = imgs('/content/tulang')

CAP = 800  # batas per kelas agar seimbang
OUT = '/content/dataset'; shutil.rmtree(OUT, ignore_errors=True)
classes = []
for c, ps in buckets.items():
    if len(ps) < 30:
        print(f'  lewati "{c}" (hanya {len(ps)} gambar)'); continue
    random.shuffle(ps); ps = ps[:CAP]
    os.makedirs(f'{OUT}/{c}', exist_ok=True)
    for i, p in enumerate(ps):
        ext = os.path.splitext(p)[1].lower() or '.jpg'
        shutil.copy(p, f'{OUT}/{c}/{c}_{i}{ext}')
    classes.append(c); print(f'  {c}: {len(ps)} gambar')
print('KELAS FINAL:', classes)
assert len(classes) >= 2, 'Butuh minimal 2 kelas.'

In [ ]:
# 6) Split train/val 85:15
import splitfolders
splitfolders.ratio('/content/dataset', output='/content/data', seed=42, ratio=(0.85, 0.15))
print('Split selesai.')

In [ ]:
# 7) Generator + bangun MobileNetV2 (base di-freeze) + latih
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
IMG, BATCH = 224, 32
tg = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_input,
        rotation_range=20, zoom_range=0.2, width_shift_range=0.1, height_shift_range=0.1, horizontal_flip=True)
vg = tf.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_input)
train = tg.flow_from_directory('/content/data/train', target_size=(IMG,IMG), batch_size=BATCH, class_mode='categorical')
val   = vg.flow_from_directory('/content/data/val',   target_size=(IMG,IMG), batch_size=BATCH, class_mode='categorical', shuffle=False)
NUM = train.num_classes
# PENTING: preprocessing [-1,1] ada di luar model (vision.js melakukannya sendiri). Jangan tambah layer Rescaling.
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG,IMG,3)); base.trainable = False
x = GlobalAveragePooling2D()(base.output); x = Dropout(0.3)(x); out = Dense(NUM, activation='softmax')(x)
model = Model(base.input, out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
cb = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
      tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3)]
model.fit(train, validation_data=val, epochs=20, callbacks=cb)

In [ ]:
# 8) (Opsional) Fine-tuning 30 layer teratas
base.trainable = True
for l in base.layers[:-30]: l.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train, validation_data=val, epochs=5, callbacks=cb)

In [ ]:
# 9) Evaluasi
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
labels = [k for k, _ in sorted(train.class_indices.items(), key=lambda x: x[1])]
p = model.predict(val); yp = p.argmax(1); yt = val.classes
print(classification_report(yt, yp, target_names=labels))
print('Confusion matrix:\n', confusion_matrix(yt, yp))

In [ ]:
# 10) Simpan model + labels.json (ARRAY urut indeks — kontrak vision.js)
import json
model.save('/content/jalari_vision.h5')
json.dump(labels, open('/content/labels.json', 'w'))
print('labels.json =', labels)

In [ ]:
# 11) Konversi ke TensorFlow.js (Python API + regenerasi labels)
import os, json, tensorflow as tf, tensorflowjs as tfjs
print('TF', tf.__version__, '| tfjs', tfjs.__version__)

# labels = urutan folder kelas (identik dengan class_indices Keras = alfabetis)
labels = sorted([d for d in os.listdir('/content/data/train') if os.path.isdir('/content/data/train/'+d)])
print('labels =', labels)

# pakai model di memori; kalau runtime sempat restart, muat dari .h5
try:
    model
except NameError:
    model = tf.keras.models.load_model('/content/jalari_vision.h5')

os.makedirs('/content/model', exist_ok=True)
try:
    tfjs.converters.save_keras_model(model, '/content/model', quantization_dtype_map={'uint8': '*'})
except TypeError:
    tfjs.converters.save_keras_model(model, '/content/model')   # versi lama tanpa kuantisasi
json.dump(labels, open('/content/model/labels.json', 'w'))
print('Isi /content/model ->', os.listdir('/content/model'))

In [ ]:
# 12) Zip + unduh -> ekstrak ke public/model/ di repo JALARI
import shutil
shutil.make_archive('/content/jalari_model', 'zip', '/content/model')
from google.colab import files
files.download('/content/jalari_model.zip')

## Menyambungkan ke Fase 3 (frontend yang sudah ada)
1. Ekstrak `jalari_model.zip` → salin `model.json`, `group1-shard*.bin`, `labels.json` ke **`public/model/`** di repo JALARI.
2. Tes lokal: `cd public && python -m http.server 8000` → buka `http://localhost:8000/#/vision`.
3. `vision.js` mendeteksi `model/model.json` (via `fetch` HEAD) dan otomatis berhenti memakai MobileNet demo — kini memakai model 4-kelas kamu.
4. Deploy: `firebase deploy --only hosting`.

**Checklist kompatibilitas:** Dense = jumlah kelas = panjang `labels.json` · tidak ada layer preprocessing di model · konversi `--input_format keras` · `labels.json` berupa array (mis. `["buah","daging","sayur","tulang"]`).